# **Predictive Text, Part II: Neural Language Models**

N-gram models are an efficient and effective way to make good predictions. However, they can only consider so many previous words.

For example, given the sentence

> First, the woman spread the jelly, and after that ...

our n-gram model might suggest *he* as the next word, because it can only consider the previous $n-1$ words. We'd like a smarter model that can look at many words.

### How do we do it?

We will build a **neural language model** using machine learning. Current state-of-the-art neural language models include [ChatGPT](https://chat.openai.com), but we will build a much smaller model that needs far less data.
***

## **Data Preparation**
### Load the corpus
We will use Spanish speech transcriptions from the [CALLHOME Corpus](https://catalog.ldc.upenn.edu/LDC96T17), or your own language.

In [59]:
# Download the helper code for this course from GitHub
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/lecs-lab/belt/main/notebooks/en/util.py",
    "util.py")

import util

# Download the sample corpus from GitHub. If you're using your own
# corpus, upload it and change this to the correct directory instead.
corpus_directory = util.download_corpus("esp/CALLHOME-fixed")

corpus = util.load_raw_text(corpus_directory)
corpus = corpus[:100_000]
print(corpus[:1000])

No porque ya no estoy tomando clases ahora. Es eso lo que estaba esperando, estaba primero tomando clases en el verano, pero mi ayuda económica para el verano no fue muy buena, entonces las tuve que dejar las clases porque si no no voy a tener dinero suficiente para pagar Grecia.
Ajá.
Entonces no, no, dejé de tomar clases y ahora no me puedo ir en la Primavera porque si no no acabo a tiempo, no me gradúo a tiempo. Entonces estaba esperando, eso era lo que estaba esperando para saber si me iba a ir a Grecia o si empezaba para ver si para de hacer planes para la Primavera y eso, pero si ya está todo lo de Grecia yo creo que mañana voy a la oficina
para ver como están los documentos y demás, lo que pasa es que como me he cambiado de casa no no me llega nada y no se nada.
Mmm.
Entonces voy a ir a ver como está todo el asunto, ya lo único que faltaría sería mi examen médico y que manden las fotos de mi pasaporte, para mi pasaporte
Bueno entonces haces allí una, una, una este, un semestre y 

Next, make text lowercase and tokenize (including ending punctuation). 

In [60]:
import re

corpus = corpus.lower()

word_regex = r"[a-záéóíúñü]+|[.?!]+"
def tokenize(text: str):
    return re.findall(word_regex, text)
    
words = tokenize(corpus)

print(words[:100])

['no', 'porque', 'ya', 'no', 'estoy', 'tomando', 'clases', 'ahora', '.', 'es', 'eso', 'lo', 'que', 'estaba', 'esperando', 'estaba', 'primero', 'tomando', 'clases', 'en', 'el', 'verano', 'pero', 'mi', 'ayuda', 'económica', 'para', 'el', 'verano', 'no', 'fue', 'muy', 'buena', 'entonces', 'las', 'tuve', 'que', 'dejar', 'las', 'clases', 'porque', 'si', 'no', 'no', 'voy', 'a', 'tener', 'dinero', 'suficiente', 'para', 'pagar', 'grecia', '.', 'ajá', '.', 'entonces', 'no', 'no', 'dejé', 'de', 'tomar', 'clases', 'y', 'ahora', 'no', 'me', 'puedo', 'ir', 'en', 'la', 'primavera', 'porque', 'si', 'no', 'no', 'acabo', 'a', 'tiempo', 'no', 'me', 'gradúo', 'a', 'tiempo', '.', 'entonces', 'estaba', 'esperando', 'eso', 'era', 'lo', 'que', 'estaba', 'esperando', 'para', 'saber', 'si', 'me', 'iba', 'a', 'ir']


Now we'd like to create a lexicon. However, unlike the previous projects, we'd like to skip words that occur only one time (in order to make our model more efficient).

#### **Exercise 1**
Modify the following code so that the lexicon only contains words that occur more than one time. We can do that by keeping two different sets, one that tracks all words (`all_words`), and one that tracks words that occur more than one time (`repeated_words`).

<details>
  <summary>Hint</summary>
    As you loop over the words, check if a word has already been added to <code>all_words</code>. If so, we can also add the word to <code>repeated_words</code>. 
</details>
<br />
<details>
  <summary>Show answer</summary>
      <pre style="background-color: honeydew; padding: 10px; border-radius: 5px;"><code style="background: none;"># All unique words in the corpus
all_words = set()

\# Just unique words that occur at least twice
repeated_words = set()

for word in words:
    # TODO: Add words that occur more than once to `repeated_words`
    if word in all_words:
        repeated_words.add(word)<br/>
    # TODO: Add each word to `all_words`
    all_words.add(word)</code></pre></details>

In [62]:
# All unique words in the corpus
all_words = set()

# Just unique words that occur at least twice
repeated_words = set()

for word in words:
    # TODO: Add words that occur more than once to `repeated_words`
    if word in all_words:
        repeated_words.add(word)

    # TODO: Add each word to all_words
    all_words.add(word)



print(f"You have {len(repeated_words)} unique words that appear more than once and {len(all_words)} total unique words.")
print(f"Should have 1229 unique words that appear more than once and 2687 total unique words.")
if len(repeated_words) == 1233 and len(all_words) == 2717:
    print("✅ Looks good!")
else:
    print("☹️ Keep trying")

You have 1233 unique words that appear more than once and 2717 total unique words.
Should have 1229 unique words that appear more than once and 2687 total unique words.
✅ Looks good!


### Split into sentences
To train our model, we will feed it one sentence at a time. Right now, `words` contains a list of strings, where each string is a word or punctuation marker. Let's divide the `words` list into a list of list of strings, where each list of strings is a sentence.

#### **Exercise 2**
Modify the following code to split `words` into sentences, using `.`, `?`, and `!` as the ending punctuation markers.


<details>
  <summary>Show answer</summary>
      <pre style="background-color: honeydew; padding: 10px; border-radius: 5px;"><code style="background: none;">all_sentences: list[list[str]] = []<br/>
current_sentence: list[str] = []<br/>
for word in words:    
    if word in ['.', '!', '?'] and len(current_sentence) > 0:
        # TODO: If the word is a punctuation mark (.!?) add the current sentence to the list of sentences and start a new sentence
        all_sentences.append(current_sentence)
        current_sentence = []
    else:
        # TODO: Else, add the word to the current sentence
        current_sentence.append(word)<br/><br/>
print("The first ten sentences:", all_sentences[:10])
print(f"The longest sentence is {max([len(sent) for sent in all_sentences])} words")</code></pre></details>

In [63]:
all_sentences: list[list[str]] = []

current_sentence: list[str] = []

for word in words:    
    if word in ['.', '!', '?'] and len(current_sentence) > 0:
        # TODO: If the word is a punctuation mark (.!?) add the current sentence to the list of sentences and start a new sentence
        all_sentences.append(current_sentence)
        current_sentence = []
    else:
        # TODO: Else, add the word to the current sentence
        current_sentence.append(word)


print("The first three sentences:", all_sentences[:3])
print(f"The longest sentence is {max([len(sent) for sent in all_sentences])} words")

The first three sentences: [['no', 'porque', 'ya', 'no', 'estoy', 'tomando', 'clases', 'ahora'], ['es', 'eso', 'lo', 'que', 'estaba', 'esperando', 'estaba', 'primero', 'tomando', 'clases', 'en', 'el', 'verano', 'pero', 'mi', 'ayuda', 'económica', 'para', 'el', 'verano', 'no', 'fue', 'muy', 'buena', 'entonces', 'las', 'tuve', 'que', 'dejar', 'las', 'clases', 'porque', 'si', 'no', 'no', 'voy', 'a', 'tener', 'dinero', 'suficiente', 'para', 'pagar', 'grecia'], ['ajá']]
The longest sentence is 193 words


### Filter long sentences
Training a machine learning model can take a long time. In order to train models more efficiently, let's filter out the long sentences in our corpus. 

#### **Exercise 3**
Modify the following code to filter sentences that are longer than 50 words.


<details>
  <summary>Show answer</summary>
      <pre style="background-color: honeydew; padding: 10px; border-radius: 5px;"><code style="background: none;">filtered_sentences = []<br/>
# TODO: Loop over the sentences in `all_sentences` and add all of the sentences <= 50 words to `filtered_sentences`
for sentence in all_sentences:
    if len(sentence) <= 50:
        filtered_sentences.append(sentence)

print(f"After filtering, we have {len(filtered_sentences)} sentences, filtering out {len(all_sentences) - len(filtered_sentences)}.")</code></pre></details>

        
<details>
  <summary>Show alternate answer</summary>
      <pre style="background-color: honeydew; padding: 10px; border-radius: 5px;"><code style="background: none;">filtered_sentences = []<br/>
# TODO: Loop over the sentences in `all_sentences` and add all of the sentences <= 50 words to `filtered_sentences`
filtered_sentences = [sent for sent in all_sentences if len(sent) <= 50]<br/>
print(f"After filtering, we have {len(filtered_sentences)} sentences, filtering out {len(all_sentences) - len(filtered_sentences)}.")</code></pre></details>

In [64]:
filtered_sentences = []

# TODO: Loop over the sentences in `all_sentences` and add all of the sentences <= 150 words to `filtered_sentences`
filtered_sentences = [sent for sent in all_sentences if len(sent) <= 50]

print(f"After filtering, we have {len(filtered_sentences)} sentences, filtering out {len(all_sentences) - len(filtered_sentences)}.")

After filtering, we have 1462 sentences, filtering out 46.


## **Building a Model**
We will build a simple language model using [Keras](https://keras.io/keras_3/). Keras is a framework that makes it easy to design ML models without any math.

We are using an **auto-regressive language model**. The model works by taking in all of the preceding words, such as

> First, the woman spread the jelly, and after that

and predicting the next word, such as `she`.

In [65]:
import os
import keras

### Preprocessing
Before we can pass strings to our model, we need to do some preprocessing. First, we need a layer that converts each word into a unique number, based on the vocabulary you defined earlier.

In [66]:
# Convert the lexicon set to a list
lexicon = sorted(repeated_words)

# Turn text inputs into a list of numbers, with a unique number for each word
string_lookup_layer = keras.layers.StringLookup(mask_token="<PAD>", vocabulary=["<START>"] + lexicon)

print(filtered_sentences[0])
string_lookup_layer(filtered_sentences[0])

['no', 'porque', 'ya', 'no', 'estoy', 'tomando', 'clases', 'ahora']


<tf.Tensor: shape=(8,), dtype=int64, numpy=array([ 770,  891, 1221,  770,  429, 1100,  192,   29])>

#### Padding
Next, we will use **padding** to make all of the sentences the same length (50 words), by adding 0s to the end. We also add a **start token**, `<START>`, to each sentence.

In [67]:
import keras_nlp

# Pad sequences to all be the same length
sentence_packer = keras_nlp.layers.StartEndPacker(sequence_length=50, start_value=2)
sentence_packer(string_lookup_layer(filtered_sentences[0]))

<tf.Tensor: shape=(50,), dtype=int64, numpy=
array([   2,  770,  891, 1221,  770,  429, 1100,  192,   29,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0])>

Now, let's run the lookup and padding layers over all the data in our corpus.

In [68]:
from tqdm.notebook import tqdm
import tensorflow

model_inputs = [sentence_packer(string_lookup_layer(sent)) for sent in tqdm(filtered_sentences)]
model_inputs = tensorflow.convert_to_tensor(model_inputs)

print(model_inputs)

  0%|          | 0/1462 [00:00<?, ?it/s]

tf.Tensor(
[[   2  770  891 ...    0    0    0]
 [   2  392  402 ...    0    0    0]
 [   2   33    0 ...    0    0    0]
 ...
 [   2 1051 1223 ...    0    0    0]
 [   2 1098    1 ...    0    0    0]
 [   2  364 1223 ...    0    0    0]], shape=(1462, 50), dtype=int64)


### Embedding
A common approach in NLP is to use an **embedding** layer to convert word tokens into vectors. These vectors can learn relationships between related words. We will use this embedding layer as the first step in our model.

In [78]:
embedder = keras.layers.Embedding(input_dim=string_lookup_layer.vocabulary_size(), output_dim=128, mask_zero=True)

embedder(sentence_packer(string_lookup_layer(filtered_sentences[0])))

<tf.Tensor: shape=(50, 128), dtype=float32, numpy=
array([[-0.04335212, -0.03702725, -0.00365837, ..., -0.00751974,
         0.03392296, -0.0498817 ],
       [ 0.02540492,  0.02094268, -0.00712878, ..., -0.04174747,
         0.02683711, -0.04717484],
       [-0.02082582, -0.03701134, -0.00950722, ...,  0.01051508,
        -0.01449858,  0.02855264],
       ...,
       [-0.02846513, -0.03433788, -0.01583491, ...,  0.02647568,
        -0.00690512,  0.00833748],
       [-0.02846513, -0.03433788, -0.01583491, ...,  0.02647568,
        -0.00690512,  0.00833748],
       [-0.02846513, -0.03433788, -0.01583491, ...,  0.02647568,
        -0.00690512,  0.00833748]], dtype=float32)>

### Assembling the model
We are using a [Long Short-Term Memory](https://en.wikipedia.org/wiki/Long_short-term_memory) model. LSTMs are a type of recurrent neural network that can handle sequences of data. Don't worry too much about how the model works, but you're welcome to play around with the parameters below.

Let's assemble the components you've seen so far, plus our LSTM layers, to a single model.

In [79]:
# Build the model
model = keras.Sequential()

# The model takes strings as inputs
model.add(keras.Input(shape=(50,)))

# Embed words into vectors
model.add(embedder)

# Add LSTM layers
model.add(keras.layers.LSTM(256, return_sequences=True))
model.add(keras.layers.Dropout(0.1))
model.add(keras.layers.LSTM(128, return_sequences=True))
model.add(keras.layers.Dropout(0.1))
# model.add(keras.layers.LSTM(128, return_sequences=True))
# model.add(keras.layers.Dropout(0.1))

# Predict the next word
output_layer = keras.layers.Dense(string_lookup_layer.vocabulary_size(), activation = "softmax")
model.add(keras.layers.TimeDistributed(output_layer))

# Compile our model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape              ┃    Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 50, 128)           │    158,208 │
├─────────────────────────────────┼───────────────────────────┼────────────┤
│ lstm_13 (LSTM)                  │ (None, 50, 256)           │    394,240 │
├─────────────────────────────────┼───────────────────────────┼────────────┤
│ dropout_3 (Dropout)             │ (None, 50, 256)           │          0 │
├─────────────────────────────────┼───────────────────────────┼────────────┤
│ lstm_14 (LSTM)                  │ (None, 50, 128)           │    197,120 │
├─────────────────────────────────┼───────────────────────────┼────────────┤
│ dropout_4 (Dropout)             │ (None, 50, 128)           │          0 │
├─────────────────────────────────┼───────────────────────────┼────────────┤
│ time_distributed_5              │ (None, 50, 1236)          │    159,444 │
│ (TimeDistributed)               │                           │            │
└─────────────────────────────────┴───────────────────────────┴────────────┘

 Total params: 909,012 (3.47 MB)

 Trainable params: 909,012 (3.47 MB)

 Non-trainable params: 0 (0.00 B)

## **Training our Model**
### Creating training labels
We'd like to train this model to predict the next word, given some sequence of words. To do that, we'll create training labels, which consist of the same words as the inputs, but shifted left one. 

For example, given the sentence:

In [80]:
print(filtered_sentences[0])

['no', 'porque', 'ya', 'no', 'estoy', 'tomando', 'clases', 'ahora']


Which is encoded as:

In [81]:
sentence_packer(string_lookup_layer(filtered_sentences[0]))

<tf.Tensor: shape=(50,), dtype=int64, numpy=
array([   2,  770,  891, 1221,  770,  429, 1100,  192,   29,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0])>

We can remove the start token and add another 0 to the end. 

In [82]:
shifted_sentence_packer = keras_nlp.layers.StartEndPacker(sequence_length=50)
shifted_sentence_packer(string_lookup_layer(filtered_sentences[0]))

<tf.Tensor: shape=(50,), dtype=int64, numpy=
array([ 770,  891, 1221,  770,  429, 1100,  192,   29,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0])>

We'll use this for training. Our model will look at the first symbol in the input, `2`, and learn to predict the first symbol in the labels, `7319`. Then, the model will look at the first *two symbols*, `2 7319`, and learn to predict the second symbol in the labels, `8386`. We repeat until we reach the end of the sentence, then move to the next sentence.

In [83]:
from tqdm import tqdm
import numpy as np

labels = np.array([shifted_sentence_packer(string_lookup_layer(sent)) for sent in tqdm(filtered_sentences)])
labels

100%|██████████| 1462/1462 [00:01<00:00, 977.40it/s] 


array([[ 770,  891, 1221, ...,    0,    0,    0],
       [ 392,  402,  646, ...,    0,    0,    0],
       [  33,    0,    0, ...,    0,    0,    0],
       ...,
       [1051, 1223,   62, ...,    0,    0,    0],
       [1098,    1,  204, ...,    0,    0,    0],
       [ 364, 1223,  253, ...,    0,    0,    0]])

## Training
Finally, it's time to train our model! The `model.fit` function trains the model to predict the `labels` from the `model_inputs`. The `epochs` parameter describes how many times we iterate over our training dataset. 

Run the cell to start training. Be aware it will take a while (around 20 minutes)!

In [84]:
history = model.fit(model_inputs, labels, epochs = 100)

Epoch 1/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 146ms/step - accuracy: 0.1165 - loss: 6.5728
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 7s 154ms/step - accuracy: 0.5545 - loss: 5.3015
Epoch 3/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 175ms/step - accuracy: 0.6304 - loss: 5.3075
Epoch 4/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 188ms/step - accuracy: 0.7040 - loss: 5.2232
Epoch 5/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 187ms/step - accuracy: 0.5954 - loss: 5.2403
Epoch 6/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 189ms/step - accuracy: 0.6660 - loss: 5.2002
Epoch 7/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 199ms/step - accuracy: 0.6734 - loss: 5.2350
Epoch 8/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 191ms/step - accuracy: 0.6297 - loss: 5.1834
Epoch 9/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 203ms/step - accuracy: 0.6656 - loss: 5.1880
Epoch 10/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 180ms/step - accuracy: 0.6790 - loss: 5.1646
Epoch 11/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 185ms/step - accuracy: 0.6968 - loss: 5.1231
Epoch 12/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 9

## **Using our Model**
Now that we have a trained model, we can use it to predict the next word. The function below will take some text, use the model to make predictions, and output the next predicted word.

In [94]:
def predict(text: str, n=1):
    # Convert text to encoded inputs
    text = util.preprocess(text)
    inputs = sentence_packer(string_lookup_layer(text))
    inputs = tensorflow.expand_dims(inputs, 0)

    # Make predictions
    predictions = model.predict(inputs, verbose=True)
    
    # Get the predictions for the last non-zero input token
    last_word_index = len(text) + 1
    prob_dist = predictions[0][last_word_index]

    choice = string_lookup_layer.get_vocabulary()[np.argmax(prob_dist)]

    # Sample from the probability distribution
    # choice = np.random.choice(string_lookup_layer.get_vocabulary(), n, p=prob_dist)
    return choice

predict("Sí , eso es para eso")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


'no'

In [50]:
def generate(prefix: str):
    length = 0
    next_token = ""
    while next_token not in ["</s>", "<PAD>"] and length < 50:
        next_token = predict(prefix)[0]
        prefix += " " + next_token
        length += 1
    return prefix

generate("yo")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━

'yo del estabas sí aquí les cariño fijo mmm así pero no a contó okey a unos tienen allí te ahí a haciendo hacer comprar es tienen lo más está llamó las ellos mandan correo r nombre [UNK] [UNK] den tienes estudiando así dos saltó de bien de noventa ah <PAD>'

In [9]:
import ipywidgets as widgets
from IPython.display import clear_output

text = widgets.Text(
    value='',
    placeholder='Start typing some text...',
    disabled=False
)
out = widgets.Output()
display(text)
display(out)

def on_change(change):
    text = change['new']
    if text[-1] == ' ':
        prediction = predict(text, 3)
        with out:
            clear_output()
            display(" | ".join(prediction))

text.observe(on_change, names=["value"])

Text(value='', placeholder='Start typing some text...')

Output()

In [106]:
model_inputs

<tf.Tensor: shape=(1559, 50), dtype=int64, numpy=
array([[   2,  375,    1, ...,    0,    0,    0],
       [   2,  832,  633, ...,    0,    0,    0],
       [   2, 1166,  375, ...,    0,    0,    0],
       ...,
       [   2, 1312,    1, ...,    0,    0,    0],
       [   2,   39, 1017, ...,    0,    0,    0],
       [   2,  648,    7, ...,    0,    0,    0]])>

In [111]:
text = util.preprocess("I")
inputs = sentence_packer(string_lookup_layer(text))
inputs = tensorflow.expand_dims(inputs, 0)
model.predict(inputs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


array([[[2.2953984e-03, 2.1109812e-02, 6.2482741e-07, ...,
         2.5831716e-05, 2.3649025e-04, 1.6469336e-05],
        [1.1083913e-06, 1.4737912e-02, 4.2069789e-07, ...,
         6.4671069e-05, 1.7982759e-04, 1.1252754e-05],
        [1.1083913e-06, 1.4737912e-02, 4.2069789e-07, ...,
         6.4671069e-05, 1.7982759e-04, 1.1252754e-05],
        ...,
        [1.1083913e-06, 1.4737912e-02, 4.2069789e-07, ...,
         6.4671069e-05, 1.7982759e-04, 1.1252754e-05],
        [1.1083913e-06, 1.4737912e-02, 4.2069789e-07, ...,
         6.4671069e-05, 1.7982759e-04, 1.1252754e-05],
        [1.1083913e-06, 1.4737912e-02, 4.2069789e-07, ...,
         6.4671069e-05, 1.7982759e-04, 1.1252754e-05]]], dtype=float32)

In [119]:
sentence_packer(string_lookup_layer(text))

<tf.Tensor: shape=(50,), dtype=int64, numpy=
array([  2, 571,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0])>